# Before vs After Metrics
Simple, reproducible notebook to compute and prove:
- Before: OpenCV template-matching baseline counts on validation images
- After: YOLOv11 fine-tuned validation metrics and counts on the same images

This notebook writes a consolidated report to `output/report_metrics.md` and appends a short notice to `REPORT.md`.


In [1]:
import os, json
from typing import List, Tuple, Dict, Any, Optional, Union
from ultralytics import YOLO
from src.yolo_utils import split_and_prepare
from src.baseline import create_template, run_baseline
import cv2
import torch


In [2]:
# Paths (absolute)
ROOT: str = os.path.abspath(os.getcwd())
XML_PATH: str = os.path.join(ROOT, 'dataset', 'annotations.xml')
IMAGES_DIR: str = os.path.join(ROOT, 'dataset', 'images')
YOLO_DATASET_DIR: str = os.path.join(ROOT, 'strawberry_dataset')
VAL_IMAGES: str = os.path.join(YOLO_DATASET_DIR, 'images', 'val')
VAL_LABELS: str = os.path.join(YOLO_DATASET_DIR, 'labels', 'val')
WEIGHTS_PATH: str = os.path.join(ROOT, 'output', 'train', 'weights', 'best.pt')
DATA_YAML: Optional[str] = os.path.join(YOLO_DATASET_DIR, 'data.yaml') if os.path.exists(os.path.join(YOLO_DATASET_DIR, 'data.yaml')) else None
TEMPLATE_PATH: str = os.path.join(ROOT, 'template', 'template.png')
OUTPUT_DIR: str = os.path.join(ROOT, 'output')
BASELINE_PRED_DIR: str = os.path.join(OUTPUT_DIR, 'baseline_eval')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(TEMPLATE_PATH), exist_ok=True)

# Prepare YOLO dataset if missing
if not os.path.exists(os.path.join(YOLO_DATASET_DIR, 'data.yaml')):
    if os.path.exists(XML_PATH) and os.path.isdir(IMAGES_DIR):
        split_and_prepare(XML_PATH, IMAGES_DIR, YOLO_DATASET_DIR)
        DATA_YAML = os.path.join(YOLO_DATASET_DIR, 'data.yaml')

device: Union[str, int] = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
print('ROOT:', ROOT)
print('DATA_YAML exists:', os.path.exists(DATA_YAML) if DATA_YAML else False)
print('WEIGHTS_PATH exists:', os.path.exists(WEIGHTS_PATH))


Device: cuda:0
ROOT: c:\Users\wbrya\OneDrive\Documents\GitHub\ripe-strawberry-detector
DATA_YAML exists: True
WEIGHTS_PATH exists: True


In [3]:
# Helpers
from typing import Iterable

def list_images(dir_path: str) -> List[str]:
    exts = ('.png', '.jpg', '.jpeg')
    if not os.path.isdir(dir_path):
        return []
    return [os.path.join(dir_path, n) for n in sorted(os.listdir(dir_path)) if n.lower().endswith(exts)]

def read_gt_count(labels_dir: str, image_path: str) -> int:
    base = os.path.splitext(os.path.basename(image_path))[0]
    lbl_path = os.path.join(labels_dir, f'{base}.txt')
    if not os.path.exists(lbl_path):
        return 0
    with open(lbl_path, 'r', encoding='utf-8') as f:
        return sum(1 for line in f if line.strip())

def baseline_predict_count(image_path: str, template_path: str, out_dir: str, threshold: float = 0.8) -> int:
    os.makedirs(out_dir, exist_ok=True)
    out_img = os.path.join(out_dir, os.path.basename(image_path).rsplit('.', 1)[0] + '.jpg')
    return run_baseline(image_path, template_path, out_img, threshold=threshold)

def yolo_predict_count(model: YOLO, image_path: str, conf: float = 0.5, device: Union[str, int] = 'cpu') -> int:
    r = model.predict(source=image_path, conf=conf, save=False, device=device)
    if not r:
        return 0
    b = getattr(r[0], 'boxes', None)
    if b is None:
        return 0
    cls_ids = getattr(b, 'cls', None)
    if cls_ids is None:
        return int(getattr(b, 'shape', [0])[0])
    return int(cls_ids.shape[0])


## Before: OpenCV Baseline

In [4]:
# Create template if missing
if os.path.exists(XML_PATH) and os.path.isdir(IMAGES_DIR) and not os.path.exists(TEMPLATE_PATH):
    create_template(XML_PATH, IMAGES_DIR, TEMPLATE_PATH)
print('Template exists:', os.path.exists(TEMPLATE_PATH), TEMPLATE_PATH)

# Evaluate baseline on up to 10 validation images
val_imgs: List[str] = list_images(VAL_IMAGES)
rows_baseline: List[Tuple[str, int, int, int]] = []
for img in val_imgs[:10]:
    gt = read_gt_count(VAL_LABELS, img)
    pr = baseline_predict_count(img, TEMPLATE_PATH, BASELINE_PRED_DIR, threshold=0.8)
    rows_baseline.append((os.path.basename(img), gt, pr, pr - gt))
rows_baseline


2025-08-26 21:51:49,283 - INFO - Baseline detection complete - 0 items found.
2025-08-26 21:51:49,305 - INFO - Baseline detection complete - 0 items found.


Template exists: True c:\Users\wbrya\OneDrive\Documents\GitHub\ripe-strawberry-detector\template\template.png


2025-08-26 21:51:49,490 - INFO - Baseline detection complete - 0 items found.
2025-08-26 21:51:49,537 - INFO - Baseline detection complete - 0 items found.
2025-08-26 21:51:49,671 - INFO - Baseline detection complete - 0 items found.
2025-08-26 21:51:49,741 - INFO - Baseline detection complete - 0 items found.
2025-08-26 21:51:49,771 - INFO - Baseline detection complete - 0 items found.
2025-08-26 21:51:49,798 - INFO - Baseline detection complete - 0 items found.


[('1.png', 24, 0, -24),
 ('13.png', 9, 0, -9),
 ('14.png', 12, 0, -12),
 ('15.png', 7, 0, -7),
 ('16.png', 9, 0, -9),
 ('21.png', 9, 0, -9),
 ('22.png', 4, 0, -4),
 ('24.png', 5, 0, -5)]

In [5]:
# Pretty print baseline table as Markdown
def to_markdown_table(rows: List[Tuple[str, int, int, int]]) -> str:
    lines = ['| filename | gt_count | pred_count | diff |', '|---|---:|---:|---:|']
    for name, gt, pr, diff in rows:
        lines.append(f'| {name} | {gt} | {pr} | {diff} |')
    return ''.join(lines)

print(to_markdown_table(rows_baseline))


| filename | gt_count | pred_count | diff ||---|---:|---:|---:|| 1.png | 24 | 0 | -24 || 13.png | 9 | 0 | -9 || 14.png | 12 | 0 | -12 || 15.png | 7 | 0 | -7 || 16.png | 9 | 0 | -9 || 21.png | 9 | 0 | -9 || 22.png | 4 | 0 | -4 || 24.png | 5 | 0 | -5 |


## After: YOLOv11 Fine-Tuned

In [6]:
metrics: Dict[str, float] = {}
if os.path.exists(WEIGHTS_PATH) and DATA_YAML and os.path.exists(DATA_YAML):
    model_for_val = YOLO(WEIGHTS_PATH)
    val_res = model_for_val.val(data=DATA_YAML, imgsz=640, device=device, project=OUTPUT_DIR, name='val-report', exist_ok=True)
    metrics = getattr(val_res, 'results_dict', {}) or {}
metrics


Ultralytics 8.3.158  Python-3.12.9 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
YOLO11s summary (fused): 100 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 2462.4253.0 MB/s, size: 1041.3 KB)


val: Scanning C:\Users\wbrya\OneDrive\Documents\GitHub\ripe-strawberry-detector\strawberry_dataset\labels\val.cache... 8 images, 0 backgrounds, 0 corrupt: 100%|██████████| 8/8 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00,  1.39it/s]


                   all          8         79      0.947      0.734      0.884      0.579
Speed: 0.8ms preprocess, 22.5ms inference, 0.0ms loss, 22.8ms postprocess per image
Results saved to c:\Users\wbrya\OneDrive\Documents\GitHub\ripe-strawberry-detector\output\val-report


{'metrics/precision(B)': np.float64(0.9466030779659758),
 'metrics/recall(B)': np.float64(0.7341772151898734),
 'metrics/mAP50(B)': np.float64(0.8842871265592139),
 'metrics/mAP50-95(B)': np.float64(0.5794882785671164),
 'fitness': np.float64(0.6099681633663261)}

In [7]:
rows_after: List[Tuple[str, int, int, int]] = []
if os.path.exists(WEIGHTS_PATH) and os.path.isdir(VAL_IMAGES):
    model_for_pred = YOLO(WEIGHTS_PATH)
    for img in val_imgs[:10]:
        gt = read_gt_count(VAL_LABELS, img)
        pr = yolo_predict_count(model_for_pred, img, conf=0.5, device=device)
        rows_after.append((os.path.basename(img), gt, pr, pr - gt))
rows_after



image 1/1 c:\Users\wbrya\OneDrive\Documents\GitHub\ripe-strawberry-detector\strawberry_dataset\images\val\1.png: 384x640 18 strawberrys, 65.6ms
Speed: 2.1ms preprocess, 65.6ms inference, 2.7ms postprocess per image at shape (1, 3, 384, 640)

image 1/1 c:\Users\wbrya\OneDrive\Documents\GitHub\ripe-strawberry-detector\strawberry_dataset\images\val\13.png: 512x640 10 strawberrys, 76.0ms
Speed: 3.4ms preprocess, 76.0ms inference, 3.2ms postprocess per image at shape (1, 3, 512, 640)

image 1/1 c:\Users\wbrya\OneDrive\Documents\GitHub\ripe-strawberry-detector\strawberry_dataset\images\val\14.png: 480x640 11 strawberrys, 66.6ms
Speed: 3.4ms preprocess, 66.6ms inference, 2.4ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 c:\Users\wbrya\OneDrive\Documents\GitHub\ripe-strawberry-detector\strawberry_dataset\images\val\15.png: 480x640 6 strawberrys, 52.5ms
Speed: 4.0ms preprocess, 52.5ms inference, 3.1ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 c:\Users\wbrya\OneD

[('1.png', 24, 18, -6),
 ('13.png', 9, 10, 1),
 ('14.png', 12, 11, -1),
 ('15.png', 7, 6, -1),
 ('16.png', 9, 9, 0),
 ('21.png', 9, 9, 0),
 ('22.png', 4, 4, 0),
 ('24.png', 5, 7, 2)]

In [8]:
print(to_markdown_table(rows_after))


| filename | gt_count | pred_count | diff ||---|---:|---:|---:|| 1.png | 24 | 18 | -6 || 13.png | 9 | 10 | 1 || 14.png | 12 | 11 | -1 || 15.png | 7 | 6 | -1 || 16.png | 9 | 9 | 0 || 21.png | 9 | 9 | 0 || 22.png | 4 | 4 | 0 || 24.png | 5 | 7 | 2 |


## Write consolidated report to `output/report_metrics.md`

In [9]:
report_out = os.path.join(OUTPUT_DIR, 'report_metrics.md')
os.makedirs(os.path.dirname(report_out), exist_ok=True)
with open(report_out, 'w', encoding='utf-8') as f:
    f.write('### Autogenerated results\n\n')
    f.write('Before (OpenCV baseline) – count accuracy on validation images:\n\n')
    f.write(to_markdown_table(rows_baseline) + '\n\n')
    f.write('After (YOLOv11 finetuned) – validation metrics:\n\n')
    f.write('```json\n')
    f.write(json.dumps(metrics, indent=2))
    f.write('After (YOLOv11 finetuned) – count accuracy on validation images:\n\n')
    f.write(to_markdown_table(rows_after) + '\n')
report_out


'c:\\Users\\wbrya\\OneDrive\\Documents\\GitHub\\ripe-strawberry-detector\\output\\report_metrics.md'

## Append a pointer into REPORT.md

In [10]:
report_md_path = os.path.join(ROOT, 'REPORT.md')
notice = ('\n\n## Before vs After (autogenerated)\n'
          '- Run `before_after_metrics.ipynb` to regenerate. Full tables/metrics are saved to `output/report_metrics.md`.\n')
if os.path.exists(report_md_path):
    with open(report_md_path, 'r', encoding='utf-8') as rf:
        content = rf.read()
    if '## Before vs After (autogenerated)' not in content:
        with open(report_md_path, 'a', encoding='utf-8') as af:
            af.write(notice)
report_md_path


'c:\\Users\\wbrya\\OneDrive\\Documents\\GitHub\\ripe-strawberry-detector\\REPORT.md'